# Evaluation — Head-to-Head and Round-Robin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/re_org/notebooks/evaluation.ipynb)

Interactive tool for comparing Connect-4 agents. Pick two or more models, set the
number of games, and hit Run. The library code lives in `src/eval.py`; this notebook
is just the control panel.

**What's included**

- All pretrained Project-1 networks (Stiles, Luke, Zan — auto-loaded from config).
- Any trained agents dropped into `RL models/`, `checkpoints/`, or any other folder
  in the repo — discovered automatically by `model_loader`.
- `random` baseline + `minimax_d1` through `minimax_d7` alpha-beta agents.

**Selection rules**

- 2 models → single head-to-head match (alternating first-player).
- 3 or more → round-robin (every pair plays every pair).
- 0 or 1 → error.

**Tactics toggle.** Each `ModelAgent` can play with tactical overrides (immediate-win,
immediate-block) **on** or **off**. Default is on (matches tournament deployment).
Turn off for pure model-only comparisons.

**Random warm-up moves.** Every `ModelAgent` + `MinimaxAgent` in this repo is
deterministic (greedy + tactics). With 0 warm-up moves, a 100-game match between two
deterministic agents reduces to 2 distinct games (one per first-player). The slider
starts at 4 to diversify starts; bump it up for more statistical resolution.

**Runs on both Colab and local.** Cell 1 detects the environment automatically.

In [ ]:
# Cell 1 — Environment setup + model loading (Colab OR local).
#
# Detects Colab vs local, clones/pulls the repo on Colab, walks up to the
# repo root on local, then initialises TF and loads every model in the
# repo (configured + auto-discovered from anywhere under the repo root).

import os
import sys
import subprocess
from pathlib import Path

GITHUB_OWNER   = 'Stiles-Clements1'
GITHUB_REPO    = 'connect4-rl-arena'
GITHUB_BRANCH  = 're_org'
GITHUB_URL     = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_ROOT = Path(f'/content/{GITHUB_REPO}')
    if not REPO_ROOT.exists():
        print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}')
        subprocess.run(
            ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        print(f'Repo already at {REPO_ROOT}; pulling latest {GITHUB_BRANCH}.')
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'fetch', '--quiet', 'origin'],
            check=False,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'checkout', '--quiet', GITHUB_BRANCH],
            check=False,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--quiet', 'origin', GITHUB_BRANCH],
            check=False,
        )
    subprocess.run(['pip', 'install', '-q', 'tqdm', 'ipywidgets'], check=False)
else:
    # Local: walk up from the notebook's CWD until we find the repo sentinel.
    here = Path.cwd().resolve()
    REPO_ROOT = None
    for p in [here, *here.parents]:
        if (p / 'src' / 'game_engine.py').exists():
            REPO_ROOT = p
            break
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find repo root starting from {here}. '
            f'Run the notebook from inside the cloned connect4-rl-arena repo.'
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Purge any previously-imported src.* modules so a re-run picks up code
# that was freshly pulled this session.
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

print(f'Running in {"Colab" if IS_COLAB else "local"} mode.')
print(f'REPO_ROOT = {REPO_ROOT}')

# GPU init BEFORE importing model_loader (otherwise models land on CPU).
import tensorflow as tf
_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass
HARDWARE = f'GPU ({_gpus[0].name.split(":")[-1]})' if _gpus else 'CPU only'
print(f'Hardware  = {HARDWARE}')

from src import model_loader
from src.eval import (
    ModelAgent, RandomAgent, MinimaxAgent,
    play_match, play_match_parallel, run_round_robin,
    format_result, round_robin_to_dataframe,
    save_results_json, save_win_rate_heatmap,
)

# load_all_models_with_discovery also scans the repo for any .keras / .h5
# files (RL models/, checkpoints/, group folders, etc.) and adds them as
# opponents. Dropping a trained model anywhere in the repo makes it
# available here on the next run — no code edits required.
models = model_loader.load_all_models_with_discovery()
print(f'\nLoaded {len(models)} models total.')

In [ ]:
# Cell 2 — Wrap every model as an Agent, plus minimax depths d1-d7.
#
# Defaults mirror tournament deployment: greedy argmax + tactical overrides
# (immediate-win / immediate-block) ON. The checkbox in Cell 3 can turn
# tactics off; when it does, the agents are rebuilt at Run time.

print(f'Hardware in use: {HARDWARE}\n')

# Canonical build: tactics ON, greedy argmax. The run handler may rebuild
# these with use_tactics=False if the toggle is flipped.
def build_agents(use_tactics: bool) -> dict:
    # Pass `name=name` so the agent's display name matches the dict key
    # (and therefore the checkbox label in cell 3). Without this the
    # progress bars and result tables show wrapper.name instead — e.g.
    # 'm1' becomes 'Stiles Transformer (M1)', which looks like a
    # different agent at a glance.
    out = {
        name: ModelAgent(wrapper, name=name, greedy=True, use_tactics=use_tactics)
        for name, wrapper in models.items()
    }
    out['random'] = RandomAgent()
    # Calibrated non-neural baselines at every depth 1-7.
    # Depth 6+ is slow on CPU (~200 ms / move); use sparingly for large
    # round-robins. `greedy` and `use_tactics` don't apply to minimax.
    for d in range(1, 8):
        out[f'minimax_d{d}'] = MinimaxAgent(depth=d)
    return out

agents = build_agents(use_tactics=True)

print(f'Built {len(agents)} agents:')
for name in agents:
    print(f'  - {name}')

In [ ]:
# Cell 3 — Interactive evaluation UI.
#
# Select 2+ agents, set the knobs, hit Run. Results are printed inline
# and also saved to logs/ (raw JSON) and report/figures/ (heatmap PNG).

import ipywidgets as widgets
from IPython.display import display, clear_output

# One checkbox per agent
checkboxes = {
    name: widgets.Checkbox(value=False, description=name, indent=False)
    for name in agents
}

n_games_slider = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description='N games per pair:',
    style={'description_width': 'initial'},
    continuous_update=False,
)

# Warm-up moves slider, default 4 (recommended minimum), range 4-16.
# 0 is intentionally NOT exposed here — two deterministic agents from an
# empty board produce only 2 distinct games, and win rates collapse to
# 0% / 50% / 100%. Start at 4 for continuous win rates; bump higher for
# more statistical resolution on close matchups.
random_init_slider = widgets.IntSlider(
    value=4, min=4, max=16, step=1,
    description='Random init moves (start-position diversifier):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
    continuous_update=False,
)

# Tactical overrides toggle. ON = agent plays an immediate win if one
# exists, blocks an immediate loss. OFF = pure model output. ON matches
# tournament deployment; OFF is for analysing the model's raw behaviour.
tactics_toggle = widgets.Checkbox(
    value=True,
    description='Tactical overrides ON (win/block) — mirrors tournament deployment',
    indent=False,
    layout=widgets.Layout(width='560px'),
)

run_button = widgets.Button(
    description='Run evaluation',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='200px'),
)

output = widgets.Output()

def on_run_clicked(b):
    with output:
        clear_output()
        selected = [name for name, cb in checkboxes.items() if cb.value]
        n = len(selected)

        if n < 2:
            print(f'ERROR: select at least 2 models ({n} currently selected).')
            return

        # Echo the chosen participants so it is unambiguous which
        # agents are running in this match. Cells 1 and 2 print every
        # model in the repo while loading; this line shows the subset
        # that the user actually selected.
        print(f'Selected ({n}): ' + ', '.join(selected) + '\n')

        n_games     = n_games_slider.value
        random_init = random_init_slider.value
        tactics_on  = tactics_toggle.value

        # Rebuild ModelAgents with the chosen tactics setting. Minimax and
        # random don't respect use_tactics — they keep their definitions.
        active_agents = build_agents(use_tactics=tactics_on)

        tactics_label = 'ON' if tactics_on else 'OFF'

        if n == 2:
            a, b_ = selected
            print(f'Head-to-head: {a} vs {b_} ({n_games} games, '
                  f'random_init={random_init}, tactics={tactics_label}, {HARDWARE})\n')
            result = play_match_parallel(
                active_agents[a], active_agents[b_],
                n_games=n_games,
                random_init_moves=random_init,
            )
            print(format_result(result))

            tag  = f'{a}_vs_{b_}_{n_games}g_init{random_init}_tac{tactics_label}'
            meta = {
                'hardware': HARDWARE, 'n_games': n_games,
                'random_init_moves': random_init,
                'use_tactics': tactics_on, 'mode': 'head_to_head',
            }
            json_path = save_results_json(result, tag=tag, metadata=meta)
            print(f'\nSaved:  {json_path}')
        else:
            print(f'Round-robin: {n} agents, {n_games} games per pair, '
                  f'random_init={random_init}, tactics={tactics_label} ({HARDWARE})\n')
            subset = {k: active_agents[k] for k in selected}
            results = run_round_robin(
                subset,
                n_games=n_games,
                parallel=True,
                random_init_moves=random_init,
            )
            df = round_robin_to_dataframe(results, selected)

            print("Win-rate matrix — row agent's win rate when playing column agent:")
            print((df * 100).round(1).fillna('--').to_string())
            print()

            ranking = df.mean(axis=1).sort_values(ascending=False)
            print('Overall ranking (mean win rate across all opponents):')
            for i, (name, wr) in enumerate(ranking.items(), 1):
                print(f'  {i}. {name:30s}  {wr:.1%}')

            tag  = f'roundrobin_{n}agents_{n_games}g_init{random_init}_tac{tactics_label}'
            meta = {
                'hardware': HARDWARE, 'n_games': n_games,
                'random_init_moves': random_init,
                'use_tactics': tactics_on, 'mode': 'round_robin',
                'agents': selected,
            }
            json_path = save_results_json(results, tag=tag, metadata=meta)
            png_path  = save_win_rate_heatmap(
                df,
                title=f'Round-robin win rates ({n_games} games/pair, {n} agents, tactics={tactics_label})',
                tag=tag,
            )
            print(f'\nSaved:  {json_path}')
            print(f'Heatmap: {png_path}')

run_button.on_click(on_run_clicked)

ui = widgets.VBox([
    widgets.HTML('<h3>Select 2 or more agents</h3>'),
    widgets.VBox(list(checkboxes.values())),
    widgets.HTML('<br>'),
    n_games_slider,
    random_init_slider,
    tactics_toggle,
    run_button,
    widgets.HTML('<hr>'),
    output,
])
display(ui)

In [ ]:
# Cell 4 (optional) — Programmatic use, no UI.
#
# Useful for scripting a specific matchup or saving results to disk.
# Example: compare the SAC model to every other agent in one round-robin.

# sac_agent = agents.get('sac_zan')          # available if RL models/sac_zan.keras is present
# if sac_agent is not None:
#     subset = {'sac_zan': sac_agent,
#               'stiles_transformer_orig': agents['stiles_transformer_orig'],
#               'zan_cnn':    agents.get('zan_cnn', agents['random']),
#               'minimax_d3': agents['minimax_d3'],
#               'minimax_d5': agents['minimax_d5'],
#               'random':     agents['random']}
#     results = run_round_robin(subset, n_games=100, parallel=True,
#                               random_init_moves=4)
#     df = round_robin_to_dataframe(results, list(subset.keys()))
#     print((df * 100).round(1).fillna('--').to_string())